## 1. 文件在系统里的位置与角色

- 文件路径：
  - `src/rmu_gazebo_simulator/rmu_gazebo_simulator/scripts/referee_system/simple_competition_1v1.py`
- 运行方式（典型由 launch 启动）：
  - `rmu_gazebo_simulator/launch/referee_system.launch.py` 会启动桥接与该脚本
- 核心职责：
  1) 维护两台标准机器人（红/蓝）的血量、弹量、命中、存活状态等
  2) 订阅攻击/射击/位姿/射频等信息，更新“对抗结果”
  3) 通过裁判命令 `RefereeCmd` 驱动比赛流程：自检、开始、停止、准备、复活、击杀
  4) 周期性发布每台机器人 `RobotStatus`，并发布 enable_power/enable_control 让机器人节点做相应限制

## 2. 模块总览（结构图）

该脚本可以拆成 4 块：

1) 工具函数：`parse_attack_info()`
2) 计时器：`GameTimer`
3) 单机器人数据结构与发布：`StandardRobot`
4) 主系统：`SimpleRefereeSystem`（创建 ROS 通信、维护比赛状态、回调处理）

运行入口：
- `main()`：创建 `rclpy` 节点 `Node('referee_system')`，实例化 `SimpleRefereeSystem(node)`，然后 `spin()`

## 3. 接口一览（Topics / Service）

### 3.1 订阅（Subscribers）
- `/referee_system/rfid_info` (`rmoss_interfaces/msg/RfidStatusArray`)
- `/referee_system/attack_info` (`std_msgs/msg/String`)
- `/referee_system/shoot_info` (`std_msgs/msg/String`)
- `/referee_system/pose_info` (`tf2_msgs/msg/TFMessage`)
- `/referee_system/referee_cmd` (`rmoss_interfaces/msg/RefereeCmd`)

### 3.2 发布（Publishers）
对每台机器人（如 `red_standard_robot1`）：
- `/referee_system/<robot>/robot_status` (`rmoss_interfaces/msg/RobotStatus`)
- `/referee_system/<robot>/enable_control` (`std_msgs/msg/Bool`)
- `/referee_system/<robot>/enable_power` (`std_msgs/msg/Bool`)

全局：
- `/referee_system/set_pose` (`geometry_msgs/msg/TransformStamped`)

### 3.3 服务（Service）
- `/exchange_ammo` (`rmoss_interfaces/srv/ExchangeAmmon`)
  - 用资源兑换弹丸上限（注意：实现里存在一个“资源扣减未写回”的逻辑点，见后文）

## 4. `parse_attack_info()`：攻击字符串解析

函数签名：`parse_attack_info(attack_str) -> tuple | None`

输入：`attack_str`（逗号分隔的字符串，来自 `/referee_system/attack_info`）

解析逻辑（关键点）：
- `attack_str.split(',')`，至少需要 2 段：
  - `info[0]`：射手信息（再按 `/` 分割，必须正好 2 段）
    - `shooter_model_name = shooter[0]`
    - `shooter_name = shooter[1]`（解析出来但后续基本没用到）
  - `info[1]`：命中目标信息（按 `/` 分割，必须正好 4 段）
    - `target_model_name = target[1]`
    - `target_link_name = target[2]`
    - `target_collision_name = target[3]`
- 只接受 `target_collision_name == 'target_collision'` 的命中

返回值：`(shooter_model_name, shooter_name, target_model_name, target_link_name)`

注意：这里对输入格式非常严格，任何字段数不满足都直接 `None`。

In [ ]:
# 可选：用脚本同样的规则，试几个 parse_attack_info 的例子
def parse_attack_info(attack_str):
    info = attack_str.split(',')
    if len(info) < 2:
        return None
    shooter = info[0].split('/')
    if len(shooter) != 2:
        return None
    shooter_model_name = shooter[0]
    shooter_name = shooter[1]
    target = info[1].split('/')
    if len(target) != 4:
        return None
    target_model_name = target[1]
    target_link_name = target[2]
    target_collision_name = target[3]
    if target_collision_name != 'target_collision':
        return None
    return shooter_model_name, shooter_name, target_model_name, target_link_name

tests = [
    'red_standard_robot1/gun,xx/blue_standard_robot1/armor_0/target_collision',
    'red_standard_robot1/gun,xx/blue_standard_robot1/armor_0/other_collision',
    'badformat',
    'red_standard_robot1/gun,xx/blue_standard_robot1/armor_0',
]
for s in tests:
    print(s, '=>', parse_attack_info(s))

## 5. `GameTimer`：比赛计时器

这是一个“可暂停/可继续”的计时器封装：
- `start()`：如果未运行，将 `start_time` 设置为 `time.time() - elapsed_time`，实现“续跑”
- `stop()`：把当前已运行的时间固化到 `elapsed_time`
- `reset()`：清空并归零
- `get_time()`：运行中返回当前运行时长，否则返回已固化的 `elapsed_time`

该计时器被 `SimpleRefereeSystem.timer_cb()` 用来做：
- 每 30 秒加资源（逻辑见后文）

## 6. `StandardRobot`：单机器人状态与对外发布

构造：`StandardRobot(node, robot_name)`

### 6.1 维护的核心状态
- `max_hp` / `remain_hp`：最大血量与当前血量
- `total_projectiles`：总弹量上限（注意：在 `supply_projectile()` 中增加）
- `used_projectiles`：已用弹数（`consume_projectile()` 增加）
- `hit_projectiles`：命中次数（`record_hit()` 增加）
- `survive`：是否存活（裁判系统会在 hp=0 时置 False 并断电）
- `tf` / `initial_tf`：真值位姿（来自 `/referee_system/pose_info`）与第一次收到的初始位姿

### 6.2 与 ROS 通信
每台机器人都有三类 publisher：
- `.../robot_status`：周期发布 `RobotStatus`
- `.../enable_control`：裁判开关控制权
- `.../enable_power`：裁判开关电源（典型用于击杀/复活）

### 6.3 `publish_status()`
每 0.5s 调用（在 `timer_cb()` 中），填充并发布：
- `max_hp, remain_hp, total_projectiles, used_projectiles, hit_projectiles`
- 若 `tf` 已收到，则填入 `msg.gt_tf = self.tf`

## 7. `SimpleRefereeSystem`：主裁判系统

### 7.1 初始化（__init__）
构造：`SimpleRefereeSystem(node)`

1) 声明参数：
- `max_hp` 默认 500
- `initial_projectiles` 默认 100
- `initial_resources` 默认 200

2) 创建 Service：
- `/exchange_ammo`（兑换弹丸）

3) 创建订阅：
- `/referee_system/rfid_info` → `rfid_status_callback`
- `/referee_system/attack_info` → `attack_info_callback`
- `/referee_system/shoot_info` → `shoot_info_callback`
- `/referee_system/pose_info` → `pose_info_callback`
- `/referee_system/referee_cmd` → `referee_cmd_callback`

4) 创建发布：
- `/referee_system/set_pose`（在 PREPARATION 时把机器人 reset 回初始位姿）

5) 创建并注册两台机器人对象：
- `red_standard_robot1`
- `blue_standard_robot1`

6) 创建 0.5s 定时器：
- `node.create_timer(0.5, self.timer_cb)`

7) 初始化资源：
- `self.initial_resources` 取参数
- `self.red_resources = self.initial_resources`
- `self.blue_resources = self.initial_resources`

并打印：`裁判系统初始化完成`

## 8. 关键回调详解

### 8.1 `/exchange_ammo`：`handle_exchange_ammo()`

语义：机器人用资源兑换弹丸。

逻辑：
1) 根据 `request.robot_name` 是否包含 `red/blue`，选择 `resources = self.red_resources / self.blue_resources`（记录阵营）
2) 如果 `0 < request.ammo_amount <= resources`：
   - `resources -= request.ammo_amount`
   - `success=True`
   - 对该机器人调用 `supply_projectile(ammo_amount)` 增加弹丸上限
   - **将扣减写回对应阵营**（已修复原逻辑未写回的漏洞）
3) 否则失败

重要实现细节：
- 当前代码已写回 `self.red_resources/self.blue_resources`，资源会真实扣减。

### 8.2 `/referee_system/rfid_info`：`rfid_status_callback()`
只是把最新 RFID 状态保存到 `self.robots_rfid_status`，当前文件里没有进一步使用它。

### 8.3 `/referee_system/attack_info`：`attack_info_callback()`

输入：`std_msgs/String`，使用 `parse_attack_info()` 严格解析。
- 若 `self.game_over` 为 True：直接忽略
- 若解析失败：忽略
- 若命中部位 link 名包含 `armor` 且目标在 `self.robots`：目标扣血 10
- 若射手在 `self.robots`：射手命中计数 +1

### 8.4 `/referee_system/shoot_info`：`shoot_info_callback()`

输入格式：`<shooter_model>/<shooter_name>,<vel>`（vel 解析为 float）
- 若 `self.game_over` 为 True：忽略
- 射手不在 robots：忽略
- 射手 `used_projectiles += 1`
- 若 `vel > 30`：射手自身扣血 10（类似“过热/超速惩罚”语义）

### 8.5 `/referee_system/pose_info`：`pose_info_callback()`

输入：`tf2_msgs/TFMessage`，遍历 `msg.transforms`：
- 若 `obj_tf.child_frame_id` 正好是 `red_standard_robot1/blue_standard_robot1`
  - 更新该机器人 `tf`
  - 如果 `initial_tf` 还没设置，则把第一次收到的 `tf` 作为 `initial_tf`


## 9. `timer_cb()`：每 0.5 秒执行的“裁判循环”

定时器周期：0.5s。核心职责：

1) **资源增长**（每满 30 秒加 50，带防抖）：
- 使用 `current_tick = int(timer.get_time() // 30)`，仅当 `current_time >= 30` 且 `current_tick > last_resource_tick` 时发放。
- 发放后更新 `last_resource_tick = current_tick`。
- 动作：`red_resources += 50`, `blue_resources += 50`

2) **存活检查**：
- 如果某机器人 `survive==True` 且 `remain_hp==0`：
  - 发送 `enable_power(False)`
  - 标记 `survive=False`

3) **终局判断**：
- 只要任一阵营所有机器人都死亡：`game_over=True`

4) **发布机器人状态**：
- 对每台机器人调用 `publish_status()`


## 10. `referee_cmd_callback()`：比赛状态机（最重要的控制入口）

订阅：`/referee_system/referee_cmd`（`rmoss_interfaces/msg/RefereeCmd`）

它像一个小状态机/指令解释器，根据 `msg.cmd` 执行不同分支：

### 10.1 `PREPARATION`（准备/复位到初始位姿）
- `game_over=True`
- 全部机器人断电：`enable_power(False)`
- 对每台机器人：
  - **若尚未收到初始 TF，则跳过并告警**（避免 assert 崩溃）
  - 否则发布 `TransformStamped` 到 `/referee_system/set_pose`，child_frame_id=机器人名，transform=initial_tf
  - sleep 0.05s（间隔发布）
- 最后全部上电：`enable_power(True)`

### 10.2 `SELF_CHECKING`（自检/重置局面）
- `game_over=False`
- 资源重置为初始值，并重置 `last_resource_tick=-1`
- 每台机器人：
  - `reset_data()`（血量/弹量/命中/存活归位）
  - `enable_power(True)`
  - `enable_control(False)`（禁止控制）
- `timer.reset()`

### 10.3 `START_GAME`（开始比赛）
- `game_over=False`
- 每台机器人：上电 + 允许控制
- `timer.start()`

### 10.4 `STOP_GAME`（停止比赛）
- `game_over=True`
- 每台机器人断电
- `timer.stop()`

### 10.5 `KILL_ROBOT`（击杀指定机器人）
- 如果 `msg.robot_name` 在 robots：
  - 断电
  - hp=0
  - survive=False

### 10.6 `REVIVE_ROBOT`（复活指定机器人）
- 如果在 robots：
  - 上电
  - hp = max_hp
  - survive=True


## 11. 宏观行为总结（你可以把它当成规则表）

- 比赛开始前：通常执行 `SELF_CHECKING`（重置数据、禁止控制）
- 开始比赛：执行 `START_GAME`（启用控制、计时开始）
- 攻击命中（attack_info）：
  - 目标 link 含 armor → 目标 -10 HP
  - 射手命中计数 +1
- 射击（shoot_info）：
  - 射手 used_projectiles +1
  - vel>30 → 射手 -10 HP
- 每 0.5 秒：
  - 检查 hp=0 的存活机器人 → 断电并标记死亡
  - 如果任一阵营无人存活 → game_over=True
  - 发布 RobotStatus
- 每 30 秒（计时器运行时）：红蓝各 +50 资源
- PREPARATION：断电→set_pose 回初始→上电（要求 initial_tf 已捕获）

## 12. 常见坑点与排查建议（已在代码中修复的点已标注）

1) **PREPARATION 的初始位姿缺失**（已修复）
- 现象：发 PREPARATION 时如果还没收到 `/referee_system/pose_info`，旧代码会 assert 崩溃。
- 现状：现在会跳过该机器人并打印 warn，不再崩溃；最好先确认 pose_info 可用。

2) **资源兑换不扣资源**（已修复）
- 旧问题：`handle_exchange_ammo()` 扣减局部变量未写回成员，资源不会减少。
- 现状：已写回 `self.red_resources/self.blue_resources`。

3) **资源增长可能重复触发**（已修复）
- 旧问题：`%30 < 0.5` 在 timer 抖动下可能重复。
- 现状：使用 `last_resource_tick` 防抖，每满 30s 只发一次。

4) **game_over 语义**
- `game_over=True` 会屏蔽 attack/shoot/timer_cb 逻辑；STOP_GAME 会断电并停表。

5) **TF 绑定条件**
- `pose_info_callback` 只认 `child_frame_id` 恰好等于机器人名；命名不一致则 initial_tf 永远为空。


## 13. 常用命令（手工触发裁判指令/观察状态）

下面命令仅作参考，具体字段名以 `rmoss_interfaces/msg/RefereeCmd` 定义为准。

### 13.1 查看接口
```bash
source /opt/ros/humble/setup.bash
source /home/abc/rm_code/2026_1_21/ros2_ws/install/setup.bash
ros2 interface show rmoss_interfaces/msg/RefereeCmd
ros2 interface show rmoss_interfaces/msg/RobotStatus
ros2 interface show rmoss_interfaces/srv/ExchangeAmmon
```

### 13.2 观察机器人状态
```bash
ros2 topic echo /referee_system/red_standard_robot1/robot_status
```

### 13.3 兑换弹丸（service call）
```bash
ros2 service call /exchange_ammo rmoss_interfaces/srv/ExchangeAmmon 
"{robot_name: red_standard_robot1, ammo_amount: 10}"
```

### 13.4 发裁判指令
你可以用 `ros2 topic pub` 向 `/referee_system/referee_cmd` 发布：
```bash
ros2 topic pub -1 /referee_system/referee_cmd rmoss_interfaces/msg/RefereeCmd '{cmd: 2}'
```
其中 `cmd` 的数值需要对照接口常量（例如 PREPARATION/SELF_CHECKING/START_GAME...）。